# AgriSense Mini: complete college-project guide

**Project type:** a partial educational implementation of the supplied AgriSense AI project.
**Coverage:** all four major features, simplified so a student can understand and present them.
**Audience:** a beginner who knows Python variables, functions, lists and basic tables.

## Abstract
Agriculture applications often combine soil data, weather inputs and images. This project demonstrates
how three supervised machine-learning models and one image-processing algorithm can be presented
in a single Streamlit application. It uses the supplied project's synthetic datasets and illustrated
sample leaves. Its purpose is to teach the development process, not to establish field-ready accuracy.

## Problem statement and objectives
Build a small application that accepts inputs, produces explainable demonstration outputs, and
documents the complete path from raw data to evaluation and interface. We will learn classification,
regression, categorical encoding, honest evaluation, image masks and basic financial arithmetic.

## All features and what they mean
| Feature | Implemented output | Method and boundary |
|---|---|---|
| Crop recommendation | Top crop and three ranked choices | Classifier on seven soil/climate values |
| Crop information | Season, growth duration, water and soil notes | Unverified reference metadata, not model-generated knowledge |
| Fertilizer recommendation | Predicted product and ranked labels | Classifier trained on synthetic rule-generated labels |
| Nutrient comparison | N/P/K input, reference target, gap and status | Explicit arithmetic in undocumented dataset units |
| Fertilizer quantity | Total kg, 50 kg bag equivalents and rounded whole bags | User-specified rate; no automatic field dosage |
| Fertilizer reference notes | Formulation, application examples and organic alternatives | Source notes, not independently validated advice |
| Yield prediction | Tons/ha, quintals/acre and total production | Regression plus unit conversion |
| Farm economics | Revenue, cost, profit/loss and margin | Editable assumed price and cost |
| Rainfall scenarios | Outputs at -25%, entered, +25% rainfall | Model sensitivity, not a causal climate forecast |
| Leaf input | Five illustrations or PNG/JPEG upload | PIL image reading and orientation handling |
| Leaf segmentation | Leaf mask, marked mask and colour overlay | Deterministic OpenCV thresholds |
| Leaf area estimate | Marked pixels / leaf pixels, severity band and region count | Estimated colour change, not confirmed infection |
| Leaf comparison | Crop-specific example description | Explicit rule-based illustration, no diagnostic confidence |
| Evaluation | Classification reports, confusion matrices, regression errors | Actual held-out results on synthetic data |
| Learning support | Five notebooks, source comments and viva answers | What, why, code, output interpretation |

## Deliberate simplifications
Real disease classification and treatment prescriptions require labelled photographs, validation and
domain expertise; they are future extensions. The source's randomly generated disease-confidence
values are not reproduced. No live weather API, live market feed, login, database or cloud deployment
is needed. Unsupported crop categories are not silently mapped to another crop. Crop choices can
carry forward where supported, but nutrient numbers stay separate because their dataset scales differ.

## Read these notebooks in order
1. **00_project_guide.ipynb** — architecture, scope, workflow and project presentation.
2. **01_crop_recommendation.ipynb** — classification from seven numerical inputs.
3. **02_fertilizer_advice.ipynb** — category encoding, classifier and transparent reference calculations.
4. **03_yield_and_economics.ipynb** — regression, leakage prevention, unit conversion and assumptions.
5. **04_leaf_image_analysis.ipynb** — pixels, colour spaces, masks, severity and failure cases.

## Environment and imports

**What we do:** Locate the project and import the libraries used in this notebook.

**Why we do it:** Relative paths let this project work after it is moved or cloned. pandas handles tables, NumPy handles arrays, and Matplotlib draws graphs. The src package contains shared, readable code.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Open this notebook from the project folder or its notebooks folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.data import SPECS, load_data
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False})
print("Project:", ROOT.name)

Project: agrisense-college-mini


**Read the output:** The printed folder should be agrisense-college-mini. In VS Code select the project's .venv Python environment as the notebook kernel. Run cells from top to bottom.

## Inspect the three datasets

**What we do:** Load every dataset and count its rows, columns and targets.

**Why we do it:** An inventory establishes exactly what data is available before any model is built.

In [2]:
inventory = []
for kind, spec in SPECS.items():
    df = load_data(kind)
    inventory.append({
        "Module": kind, "Rows": len(df), "Columns": len(df.columns),
        "Task": spec["task"], "Target": spec["target"],
    })
display(pd.DataFrame(inventory))

,Module,Rows,Columns,Task,Target
0,crop,2200,8,classification,label
1,fertilizer,1200,9,classification,Fertilizer_Name
2,yield,2500,10,regression,Yield_tons_per_ha


**Read the output:** There are 2,200 crop, 1,200 fertilizer and 2,500 yield rows: 5,900 records in total. A CSV row is a generated example, not a verified farm observation.

## Architecture and file responsibilities
```text
CSV data -> inspect/check -> split -> encode -> compare -> train -> evaluate -> saved model
                                                                          |
user inputs -> validate ------------------------------------------------> prediction -> app result

leaf image -> resize -> HSV colour values -> leaf mask -> marked mask -> overlay + area estimate
```

| File/folder | What it does | Why it exists |
|---|---|---|
| `app.py` | Five-page Streamlit interface | Connects all four modules for demonstration |
| `src/data.py` | Dataset schemas, paths and input validation | Keeps feature definitions consistent |
| `src/ml.py` | Pipelines, selection, training, prediction, importance | Shares model logic between notebooks and app |
| `src/advisory.py` | Reference gaps, quantities, conversions and economics | Separates arithmetic assumptions from ML |
| `src/vision.py` | Deterministic leaf segmentation | Shows what image processing can and cannot measure |
| `train_models.py` | Retrains the three models | Reproduces model artifacts and metric reports |
| `models/` | Locally trained joblib bundles | Allows predictions without retraining |
| `data/` | CSVs, reference text and provenance | Preserves input data and its origin |
| `samples/` | Five illustrated leaves | Gives a repeatable image demonstration |
| `reports/` | Measured metrics and verification records | Keeps evidence separate from claims |
| `tests/` | Integration and mathematical checks | Detects regressions and incorrect edge cases |
| `tools/` | Builds/executes notebooks; prepares selected reference assets | Makes setup and documentation reproducible |
| `.vscode/` | Interpreter, debugger and extension suggestions | Makes the project easy to open in VS Code |

## Stepwise implementation
1. Establish scope; copy only the three datasets, sample images and selected reference text.
2. Inspect data quality and declare inputs/targets; document synthetic-data limitations.
3. Split data 80/20. Compare candidates by three-fold validation on training data only.
4. Fit the selected candidate on training data; evaluate once on held-out data.
5. Implement explicit arithmetic and a deterministic image-processing module.
6. Connect the modules to forms, validation, charts and clear result labels.
7. Execute the notebooks, run integration checks and demonstrate the complete workflow.

## Essential concepts
**Feature:** an input column. **Target/label:** the answer a supervised model learns to predict.
**Classification:** selecting a category. **Regression:** estimating a number.
**Training:** fitting the model from examples. **Inference:** using a fitted model on a new input.
**Validation:** choosing among candidates without using the final test data.
**Test set:** held-out examples used to measure the selected model's performance.
**Overfitting:** learning details that do not generalize. **Leakage:** using information unavailable
at prediction time, such as the answer or a direct derivative of it.
**Random seed:** makes random splits and model construction reproducible within the tested environment.

## Read measured model results

**What we do:** Load the locally trained bundles and summarize the saved test metrics.

**Why we do it:** The report must contain measured values. High scores on synthetic data do not prove field reliability.

In [3]:
from src.ml import load_model
summary = []
for kind in SPECS:
    metrics = load_model(kind)["metrics"]
    summary.append({
        "Module": kind, "Selected model": metrics["selected_model"],
        "Test rows": metrics["test_rows"],
        "Accuracy": metrics.get("accuracy", np.nan),
        "Macro F1": metrics.get("macro_f1", np.nan),
        "MAE (tons/ha)": metrics.get("mae", np.nan),
        "R2": metrics.get("r2", np.nan),
    })
display(pd.DataFrame(summary).round(4))

,Module,Selected model,Test rows,Accuracy,Macro F1,MAE (tons/ha),R2
0,crop,Random Forest,440,0.9932,0.9932,NaN,NaN
1,fertilizer,Decision Tree,240,1.0000,1.0000,NaN,NaN
2,yield,Random Forest,500,NaN,NaN,1.0977,0.9871


**Read the output:** Blank entries mean a metric is not applicable. Accuracy is for classification; MAE and R² are for regression. The leaf module has no measured disease-classification accuracy because it is not a trained disease model.

## How to run in VS Code
Open this project folder, select `.venv\Scripts\python.exe` as the Python interpreter, and choose it
as the notebook kernel. Start here, then use **Run All** in each notebook. The Python and Jupyter
extensions are recommended in `.vscode/extensions.json`.

In the terminal:
```powershell
.\.venv\Scripts\python.exe train_models.py
.\.venv\Scripts\python.exe -m streamlit run app.py --server.address 127.0.0.1
```
Alternatively use the VS Code **Run AgriSense app** debug configuration. The README contains clean-machine setup.

## Suggested demonstration sequence
1. Introduce the problem and explain why the dataset is synthetic.
2. Show a crop dataset row, its input features, and the train/test split.
3. Run a crop prediction and explain its model score.
4. Open fertilizer advice, compare reference gaps and edit the assumed rate.
5. Open yield, change assumed price/cost and explain a negative-profit result.
6. Compare a healthy leaf illustration with a spotted one; show both masks.
7. Finish with measured results, limitations and future work.

## Viva questions with short answers
**Why use Python?** Its table, ML, plotting and image-processing libraries fit this small project.

**Why Streamlit?** It creates forms, charts and result displays directly from Python.

**Why not train on all rows before measuring accuracy?** That would remove an independent test set.

**Why might the fertilizer model reach 100%?** Its synthetic labels follow simple rules that a tree can reproduce.

**Why not use deep learning for the leaf module?** The supplied archive has illustrated samples and no
labelled photographic training dataset. A small thresholding demonstration is explainable with this evidence.

**What would make the project ready for real use?** Verified data with units and provenance, geographic and
temporal evaluation, labelled image training data, calibrated uncertainty and expert agronomic validation.

## Source and attribution
Reference: user-supplied `agri-ai-suite.zip` (AgriSense AI). Dataset and sample-image hashes are in
`data/provenance.json`. Source crop/fertilizer descriptions are clearly marked. The original serialized
models were not reused. The archive did not contain a license file; confirm redistribution rights before
publishing its data/assets publicly.